# 🚀 SUSA AURA 1B — Master Google Colab Training & Verification Pipeline

> **Adaptive Universal Reasoning Assistant (SUSA AURA)**  
> **Target Architecture:** Native 1-Billion-Parameter Decoder-Only Transformer (`configs/target_1b.yaml`)  
> **Verified Parameters:** `1,009,338,368` (~1.009B)  
> **Target Environment:** Google Colab Free Tier GPU (NVIDIA Tesla T4 ~15GB VRAM) + Google Drive Persistent Storage

---

### 📌 Core Architecture & Hardware Standards:
1. **First-Class 1B Commitment:** We commit to the native 1-billion-parameter architecture from day one. We never silently shrink dimensions or substitute third-party pretrained weights.
2. **Tesla T4 (15GB VRAM) Memory Optimization:** Standard FP32 AdamW for a 1B model requires ~15.43 GiB VRAM, which triggers an Out-of-Memory (OOM) crash on a 14.56 GiB T4 GPU. We solve this by pairing **8-bit AdamW (`bitsandbytes`) with activation gradient checkpointing**, reducing memory to **~6.97 GiB VRAM** while maintaining full mathematical fidelity.
3. **Mutual Compatibility Guarantee:** Model embedding dimension ($V=32,000$), tokenizer vocabulary ($V=32,000$), and binary dataset shards (`uint16`, token IDs $< 32,000$) are strictly synchronized. The smoke tokenizer (vocab 790) is never substituted for the 1B model.
4. **Resilience & Zero Duplicate Storage:** Google Drive (`MyDrive/SUSA-AURA-1B/`) stores reusable datasets, tokenizers, checkpoints, and reports. If data is already present, it is reused immediately. If missing, it is prepared automatically and archived to Drive.

---

### 🧭 Execution Modes (Configured in Cell 1):
- **`AUDIT` (Default):** Inspects hardware, storage, code, config, tokenizer, and datasets without training.
- **`PREPARE`:** Sets up persistent Google Drive storage, acquires source code, trains the 32k tokenizer, generates binary shards, and runs an initial single-step validation test.
- **`PILOT`:** Validates 1B model execution on GPU with a single step, tests checkpoint resumption, and optionally runs a 10-step pilot upon explicit opt-in.
- **`RESUME`:** Resumes training from a validated saved checkpoint without restarting from step zero.


## Cell 1: Master Configuration & Execution Mode Selection


In [ ]:
#@title Cell 1: Master Control Panel { display-mode: "form" }
import os
import sys
import json
import time
from pathlib import Path

# ==============================================================================
# 1. EXECUTION MODE (Default: AUDIT)
# ==============================================================================
MODE = "PREPARE" #@param ["AUDIT", "PREPARE", "PILOT", "RESUME"]

# ==============================================================================
# 2. STORAGE & REPOSITORY SETTINGS
# ==============================================================================
MOUNT_DRIVE = True #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/SUSA-AURA-1B" #@param {type:"string"}
LOCAL_WORKSPACE = "/content/aura-work" #@param {type:"string"}

REPO_SOURCE = "GIT_CLONE" #@param ["GIT_CLONE", "DRIVE_ARCHIVE", "LOCAL_UPLOAD"]
GIT_URL = "https://github.com/saketh-nandu/susa-aura.git" #@param {type:"string"}
GIT_BRANCH = "main" #@param {type:"string"}

# ==============================================================================
# 3. TRAINING & PILOT CONTROLS
# ==============================================================================
PROFILE_CONFIG = "configs/target_1b.yaml" #@param {type:"string"}
USE_8BIT_ADAM = True #@param {type:"boolean"}
RUN_10_STEP_PILOT = False #@param {type:"boolean"}
RESUME_CHECKPOINT = "" #@param {type:"string"}
RESUME_ADDITIONAL_STEPS = 20 #@param {type:"integer"}

# ==============================================================================
# 4. SINGLE SOURCE OF TRUTH STATE MANAGEMENT
# ==============================================================================
AURA_SESSION_CONFIG = {
    "MODE": MODE,
    "MOUNT_DRIVE": MOUNT_DRIVE,
    "DRIVE_ROOT": DRIVE_ROOT,
    "LOCAL_WORKSPACE": LOCAL_WORKSPACE,
    "REPO_SOURCE": REPO_SOURCE,
    "GIT_URL": GIT_URL,
    "GIT_BRANCH": GIT_BRANCH,
    "PROFILE_CONFIG": PROFILE_CONFIG,
    "USE_8BIT_ADAM": USE_8BIT_ADAM,
    "RUN_10_STEP_PILOT": RUN_10_STEP_PILOT,
    "RESUME_CHECKPOINT": RESUME_CHECKPOINT.strip(),
    "RESUME_ADDITIONAL_STEPS": RESUME_ADDITIONAL_STEPS,
    "INIT_TIMESTAMP": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
}

cache_dir = Path("/content/aura-cache")
cache_dir.mkdir(parents=True, exist_ok=True)
with open(cache_dir / "session_config.json", "w", encoding="utf-8") as f:
    json.dump(AURA_SESSION_CONFIG, f, indent=2)

print("=" * 75)
print("  SUSA AURA 1B — MASTER CONTROL PANEL INITIALIZED")
print("=" * 75)
print(f"  [Active Mode]:          {MODE}")
print(f"  [Google Drive]:         {'ENABLED' if MOUNT_DRIVE else 'DISABLED'} -> {DRIVE_ROOT}")
print(f"  [Workspace]:            {LOCAL_WORKSPACE}")
print(f"  [Repository]:           {REPO_SOURCE} ({GIT_URL} @ {GIT_BRANCH})")
print(f"  [Architecture]:         {PROFILE_CONFIG} (Native 1,009,338,368 params)")
print(f"  [8-bit AdamW]:          {USE_8BIT_ADAM} (Mandatory for T4 15GB VRAM)")
print(f"  [Run 10-Step Pilot]:    {RUN_10_STEP_PILOT}")
if MODE == "RESUME":
    ckpt_target = RESUME_CHECKPOINT if RESUME_CHECKPOINT else "AUTO-DETECT (Latest valid checkpoint)"
    print(f"  [Resume Target]:        {ckpt_target} (+{RESUME_ADDITIONAL_STEPS} steps)")
print("=" * 75)

if MODE == "AUDIT":
    print("  Mode Note: Audit Mode will inspect hardware, datasets, and models without executing training.")
elif MODE == "PREPARE":
    print("  Mode Note: Prepare Mode will establish storage, auto-generate 32k datasets if missing, and run a 1-step test.")
elif MODE == "PILOT":
    print("  Mode Note: Pilot Mode will validate single-step execution, test checkpoint resume, and optionally run 10 steps.")
elif MODE == "RESUME":
    print("  Mode Note: Resume Mode will restore an existing checkpoint to continue training.")


## Cell 2: Google Drive Mounting & Persistent Folder Hierarchy Setup


In [ ]:
#@title Cell 2: Mount Google Drive & Setup Persistent Hierarchy { display-mode: "form" }
import shutil
import sys
from pathlib import Path

print(f">>> [Cell 2 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']} | Drive Target: {AURA_SESSION_CONFIG['DRIVE_ROOT']}")

IN_COLAB = "google.colab" in sys.modules

DRIVE_PATHS = {
    "root": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]),
    "datasets": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "datasets",
    "tokenizer": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "tokenizer",
    "checkpoints": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "checkpoints",
    "logs": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "logs",
    "reports": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "reports",
    "source": Path(AURA_SESSION_CONFIG["DRIVE_ROOT"]) / "source",
}

if AURA_SESSION_CONFIG["MOUNT_DRIVE"]:
    if IN_COLAB:
        from google.colab import drive
        print("Mounting Google Drive at /content/drive...")
        try:
            drive.mount("/content/drive")
            print("[✓] Google Drive successfully mounted.")
        except Exception as e:
            print(f"[!] Drive mount note: {e}")
    else:
        print("[INFO] Local environment detected. Using local path fallback.")

    # Create persistent directory hierarchy
    for name, p in DRIVE_PATHS.items():
        p.mkdir(parents=True, exist_ok=True)
        print(f"  - Directory Ready: {p}")

    # Inspect free space on Google Drive
    try:
        total_b, used_b, free_b = shutil.disk_usage(DRIVE_PATHS["root"])
        print(f"\n  Drive Capacity: {free_b / (1024**3):.2f} GiB free out of {total_b / (1024**3):.2f} GiB total.")
        if free_b < 2 * (1024**3):
            print("  [!] CAUTION: Less than 2 GiB free on Google Drive. Clean old checkpoints if needed.")
    except Exception as e:
        print(f"  [!] Note: Could not query Drive quota: {e}")
else:
    print("[INFO] Google Drive mounting disabled. Operating in local ephemeral storage only.")


## Cell 3: Local Workspace Initialization & Repository Transfer


In [ ]:
#@title Cell 3: Setup Local Workspace & Sync Codebase { display-mode: "form" }
import subprocess
import zipfile
import tarfile
from pathlib import Path

print(f">>> [Cell 3 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']} | Repo Source: {AURA_SESSION_CONFIG['REPO_SOURCE']}")

workspace_path = Path(AURA_SESSION_CONFIG["LOCAL_WORKSPACE"])
cache_path = Path("/content/aura-cache")
cache_path.mkdir(parents=True, exist_ok=True)

# Detect if current directory is already the repository
cwd = Path.cwd().resolve()
if (cwd / "aura").exists() and (cwd / "configs").exists():
    print(f"[✓] Current directory is already SUSA AURA repository: {cwd}")
    workspace_path = cwd
else:
    workspace_path.mkdir(parents=True, exist_ok=True)
    src_type = AURA_SESSION_CONFIG["REPO_SOURCE"]
    git_url = AURA_SESSION_CONFIG["GIT_URL"]
    git_branch = AURA_SESSION_CONFIG["GIT_BRANCH"]

    if src_type == "GIT_CLONE":
        if (workspace_path / "aura").exists():
            print(f"Workspace {workspace_path} already exists. Pulling latest {git_branch}...")
            subprocess.run(["git", "checkout", git_branch], cwd=str(workspace_path), check=False)
            subprocess.run(["git", "pull", "origin", git_branch], cwd=str(workspace_path), check=False)
        else:
            print(f"Cloning {git_url} (branch: {git_branch}) into {workspace_path}...")
            subprocess.run(["git", "clone", "--branch", git_branch, git_url, str(workspace_path)], check=True)
    elif src_type == "DRIVE_ARCHIVE":
        source_zip = DRIVE_PATHS["source"] / "susa-aura-latest.zip"
        source_tar = DRIVE_PATHS["source"] / "susa-aura-latest.tar.gz"
        if source_zip.exists():
            print(f"Extracting source archive from Google Drive: {source_zip}...")
            with zipfile.ZipFile(source_zip, "r") as zf:
                zf.extractall(workspace_path)
        elif source_tar.exists():
            print(f"Extracting source archive from Google Drive: {source_tar}...")
            with tarfile.open(source_tar, "r:*") as tf:
                tf.extractall(workspace_path)
        else:
            print(f"[!] No archive found in {DRIVE_PATHS['source']}. Defaulting to Git clone.")
            subprocess.run(["git", "clone", "--branch", git_branch, git_url, str(workspace_path)], check=True)

# Add workspace to sys.path and switch working directory
if str(workspace_path) not in sys.path:
    sys.path.insert(0, str(workspace_path))

os.chdir(str(workspace_path))
print(f"[✓] Working directory set to: {Path.cwd()}")

# Check Git commit hash provenance
try:
    commit_sha = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL).strip()
    branch_name = subprocess.check_output(["git", "branch", "--show-current"], text=True, stderr=subprocess.DEVNULL).strip()
    print(f"[Provenance] Git Commit: {commit_sha} (Branch: {branch_name})")
except Exception:
    print("[Provenance] Operating outside active git tree.")


## Cell 4: Install Dependencies & Verify Packaging


In [ ]:
#@title Cell 4: Install Dependencies & Verify Packaging { display-mode: "form" }
import sys
import subprocess

print(f">>> [Cell 4 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']}")

# Verify if package is installed in editable mode
try:
    import aura
    import yaml
    import psutil
    import tokenizers
    print("[✓] Core susa-aura package and dependencies are ready.")
except ImportError:
    print("Installing editable susa-aura package...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)

# Verify bitsandbytes for 8-bit AdamW
try:
    import bitsandbytes as bnb
    print("[✓] bitsandbytes is installed and ready for 8-bit AdamW.")
except ImportError:
    print("Installing bitsandbytes for 8-bit AdamW memory optimization...")
    subprocess.run([sys.executable, "-m", "pip", "install", "bitsandbytes"], check=False)


## Cell 5: Free Colab Hardware & Memory Preflight Diagnostics


In [ ]:
#@title Cell 5: Hardware Preflight & Memory Feasibility Check { display-mode: "form" }
import torch
import psutil
import shutil
from aura.utils.system_checker import SystemInfo
from aura.model.config import ModelConfig
from aura.utils.resource_estimator import estimate_resources

print(f">>> [Cell 5 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']}")
print("=" * 75)
print("  SUSA AURA 1B — HARDWARE & MEMORY PREFLIGHT AUDIT")
print("=" * 75)

# 1. System Resources
sys_ram_total = psutil.virtual_memory().total / (1024**3)
sys_ram_avail = psutil.virtual_memory().available / (1024**3)
disk_total, disk_used, disk_free = shutil.disk_usage("/content")

print(f"  Python Version:         {sys.version.split()[0]}")
print(f"  PyTorch Version:        {torch.__version__}")
print(f"  System RAM Total:       {sys_ram_total:.2f} GiB (Available: {sys_ram_avail:.2f} GiB)")
print(f"  NVMe Disk Free:         {disk_free / (1024**3):.2f} GiB of {disk_total / (1024**3):.2f} GiB")

# 2. CUDA GPU Resources
cuda_available = torch.cuda.is_available()
print(f"  CUDA GPU Available:     {cuda_available}")

gpu_name = "None (CPU only)"
gpu_vram = 0.0

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"  Assigned GPU:           {gpu_name}")
    print(f"  Total GPU VRAM:         {gpu_vram:.2f} GiB")
    print(f"  Compute Capability:     {torch.cuda.get_device_capability(0)}")
else:
    print("  [!] CAUTION: No GPU detected. To train the 1B model, enable T4 GPU in Runtime menu.")

# 3. Model 1B Resource Calculation
cfg_path = Path(AURA_SESSION_CONFIG["PROFILE_CONFIG"])
cfg = ModelConfig.from_yaml(cfg_path)
res_standard = estimate_resources(cfg, batch_size=1, seq_len=cfg.max_seq_len, use_8bit_adam=False, gradient_checkpointing=True)
res_8bit = estimate_resources(cfg, batch_size=1, seq_len=cfg.max_seq_len, use_8bit_adam=True, gradient_checkpointing=True)

print("-" * 75)
print(f"  Target 1B Model:        {cfg.profile_name} (1,009,338,368 parameters)")
print(f"  Context Window:         {cfg.max_seq_len} tokens")
print(f"  VRAM Req (Standard AdamW):  ~{res_standard.estimated_total_training_ram_gb:.2f} GiB (Exceeds T4 14.56GB VRAM -> OOM)")
print(f"  VRAM Req (8-bit AdamW):     ~{res_8bit.estimated_total_training_ram_gb:.2f} GiB (Fits in T4 14.56GB VRAM)")
print("-" * 75)

CAN_RUN_GPU = False
if not cuda_available:
    print("  [VERDICT]: BLOCKED (CPU only). 1B model step requires GPU.")
elif gpu_vram < 8.0:
    print(f"  [VERDICT]: BLOCKED. GPU VRAM ({gpu_vram:.2f} GiB) < 8.0 GiB minimum.")
elif gpu_vram < 15.0:
    if AURA_SESSION_CONFIG["USE_8BIT_ADAM"]:
        print(f"  [VERDICT]: COMPATIBLE. 8-bit AdamW enabled. Fits safely on {gpu_name} ({gpu_vram:.2f} GiB VRAM).")
        CAN_RUN_GPU = True
    else:
        print("  [VERDICT]: RISK OF OOM. Enable USE_8BIT_ADAM=True in Cell 1 to fit in 15GB VRAM.")
else:
    print(f"  [VERDICT]: EXCELLENT. {gpu_name} ({gpu_vram:.2f} GiB VRAM) has ample capacity.")
    CAN_RUN_GPU = True
print("=" * 75)


## Cell 6: Verify 1B Model Architecture & Exact Parameter Count


In [ ]:
#@title Cell 6: Validate 1B Model Architecture & Parameters { display-mode: "form" }
from aura.model.config import ModelConfig
from aura.model.transformer import AuraModel

print(f">>> [Cell 6 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']}")

cfg_path = Path(AURA_SESSION_CONFIG["PROFILE_CONFIG"])
cfg = ModelConfig.from_yaml(cfg_path)
stats = cfg.calculate_exact_parameters()

print("=" * 75)
print("  SUSA AURA 1B — ARCHITECTURE & TENSOR VERIFICATION")
print("=" * 75)
print(cfg.print_parameter_summary())

total_params = stats["total_parameters"]
assert total_params == 1_009_338_368, f"Parameter mismatch: Expected 1,009,338,368, got {total_params:,}"

print(f"[✓] Exact Parameter Count Verified: {total_params:,} parameters.")
print(f"[✓] Vocabulary Size:                 {cfg.vocab_size:,} tokens")
print(f"[✓] Tied Embeddings:                 lm_head.weight == tok_embeddings.weight")
print(f"[✓] Positional Encoding:             RoPE with theta={cfg.rope_theta}")
print(f"[✓] Attention Implementation:        GQA (16 Query heads, 8 KV heads)")
print("=" * 75)
print("[Status Notice]: Model weights are randomly initialized.")
print("                There are zero pretrained weights until pretraining is executed.")


## Cell 7: Dataset Discovery, Validation & Auto-Preparation Pipeline


In [ ]:
#@title Cell 7: Discover, Inspect & Auto-Prepare 32k Pretraining Shards { display-mode: "form" }
import json
import shutil
import subprocess
from pathlib import Path
from aura.data.dataset import MemoryMappedPretrainingDataset
from aura.data.deduplication import check_train_val_leakage

print(f">>> [Cell 7 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']}")
print("=" * 75)
print("  SUSA AURA 1B — DATASET DISCOVERY, VALIDATION & PREPARATION")
print("=" * 75)

# Target configuration requires vocab_size: 32000
TARGET_VOCAB = 32000
DATASETS_READY = False

# Search order: 1) Google Drive, 2) Local Workspace
drive_train_bin = DRIVE_PATHS["datasets"] / "train.bin"
drive_val_bin = DRIVE_PATHS["datasets"] / "val.bin"
local_train_bin = Path("data/train.bin")
local_val_bin = Path("data/val.bin")

# Check if valid 32k dataset already exists on Google Drive
if drive_train_bin.exists():
    drive_meta_path = drive_train_bin.with_suffix(".json")
    if drive_meta_path.exists():
        with open(drive_meta_path, "r", encoding="utf-8") as f:
            d_meta = json.load(f)
        if d_meta.get("vocab_size") == TARGET_VOCAB:
            print(f"[✓] Found verified 32k dataset on Google Drive: {drive_train_bin}")
            local_train_bin.parent.mkdir(parents=True, exist_ok=True)
            if not local_train_bin.exists() or local_train_bin.stat().st_size != drive_train_bin.stat().st_size:
                print("Copying dataset from Drive to fast local NVMe...")
                shutil.copy2(drive_train_bin, local_train_bin)
                shutil.copy2(drive_meta_path, local_train_bin.with_suffix(".json"))
                if drive_val_bin.exists():
                    shutil.copy2(drive_val_bin, local_val_bin)
                    shutil.copy2(drive_val_bin.with_suffix(".json"), local_val_bin.with_suffix(".json"))

# Check if local dataset is valid 32k
is_local_valid = False
if local_train_bin.exists():
    local_meta_path = local_train_bin.with_suffix(".json")
    if local_meta_path.exists():
        with open(local_meta_path, "r", encoding="utf-8") as f:
            l_meta = json.load(f)
        if l_meta.get("vocab_size") == TARGET_VOCAB:
            is_local_valid = True
            print(f"[✓] Local 32k dataset confirmed: {local_train_bin} ({l_meta.get('total_tokens', 0):,} tokens)")
        else:
            print(f"[!] Warning: Existing dataset has vocab {l_meta.get('vocab_size')} != {TARGET_VOCAB}. Re-preparation required.")

# AUTO-PREPARATION: If missing or incompatible, prepare the 32k dataset
if not is_local_valid:
    print("\n[!] 32k dataset shards missing or incompatible. Initiating automated preparation...")

    # Step 1: Ensure code corpus is present
    raw_code_dir = Path("data/raw_code")
    py_files = list(raw_code_dir.glob("**/*.py")) if raw_code_dir.exists() else []
    if len(py_files) < 10:
        print("  [1/3] Ingesting Python code corpus via scripts/download_dataset.py...")
        subprocess.run([sys.executable, "scripts/download_dataset.py", "--source", "stdlib", "--output-dir", "data/raw_code"], check=True)

    # Step 2: Ensure 32k tokenizer is trained
    tok_dir = Path("tokenizer_cache/target_32k")
    if not (tok_dir / "tokenizer.json").exists():
        print(f"  [2/3] Training 32,000-vocabulary BPE tokenizer using data/raw_code...")
        subprocess.run([
            sys.executable, "scripts/train_tokenizer.py",
            "--input-dir", "data/raw_code",
            "--vocab-size", str(TARGET_VOCAB),
            "--output-dir", str(tok_dir)
        ], check=True)

    # Step 3: Tokenize into memory-mapped binary shards
    print(f"  [3/3] Generating binary pretraining shards (train.bin & val.bin)...")
    subprocess.run([
        sys.executable, "scripts/prepare_dataset.py",
        "--tokenizer-dir", str(tok_dir),
        "--input-dir", "data/raw_code",
        "--output-dir", "data"
    ], check=True)

    # Archive to Google Drive so it is never re-prepared
    if AURA_SESSION_CONFIG["MOUNT_DRIVE"] and DRIVE_PATHS["datasets"].exists():
        print(f"  Archiving newly generated 32k dataset to Google Drive: {DRIVE_PATHS['datasets']}...")
        shutil.copy2("data/train.bin", DRIVE_PATHS["datasets"] / "train.bin")
        shutil.copy2("data/train.json", DRIVE_PATHS["datasets"] / "train.json")
        shutil.copy2("data/val.bin", DRIVE_PATHS["datasets"] / "val.bin")
        shutil.copy2("data/val.json", DRIVE_PATHS["datasets"] / "val.json")
        print("  [✓] Dataset successfully archived on Google Drive.")

# Final Validation of Shard
if local_train_bin.exists():
    ds = MemoryMappedPretrainingDataset(local_train_bin, seq_len=cfg.max_seq_len)
    num_seqs = len(ds)
    sample_seq = ds[0]
    max_id = int(sample_seq["input_ids"].max().item())
    min_id = int(sample_seq["input_ids"].min().item())

    print(f"\n  [Validation Results]:")
    print(f"  - Total Sequences:      {num_seqs:,} (seq_len={cfg.max_seq_len})")
    print(f"  - Total Tokens:         {ds.total_tokens:,}")
    print(f"  - Token ID Range:       [{min_id}, {max_id}] (Target Vocabulary: {TARGET_VOCAB:,})")

    assert max_id < TARGET_VOCAB, f"Token ID {max_id} exceeds model vocab size {TARGET_VOCAB}"
    DATASETS_READY = True
    print("[✓] Pretraining binary dataset is verified 100% compatible with 1B model!")
else:
    print("[ERROR] Dataset preparation failed to produce data/train.bin.")


## Cell 8: Tokenizer Verification & Vocabulary Check


In [ ]:
#@title Cell 8: Verify Tokenizer Vocabulary { display-mode: "form" }
from aura.tokenizer.bpe_tokenizer import AuraTokenizer

print(f">>> [Cell 8 Active] Current Mode: {AURA_SESSION_CONFIG['MODE']}")

tok_candidates = [
    Path("tokenizer_cache/target_32k"),
    DRIVE_PATHS["tokenizer"] / "target_32k",
]

found_tok_dir = next((p for p in tok_candidates if (p / "tokenizer.json").exists()), None)

if found_tok_dir:
    tok = AuraTokenizer.load(found_tok_dir)
    print(f"[✓] Target Tokenizer Loaded: {found_tok_dir}")
    print(f"    - Vocabulary Size:       {tok.vocab_size:,} tokens (Matches 1B Config: {tok.vocab_size == 32000})")
    print(f"    - Special Tokens:        PAD={tok.pad_id}, BOS={tok.bos_id}, EOS={tok.eos_id}")

    test_str = "def susa_aura_1b(parameters: int = 1009338368) -> bool:\n    return True\n"
    enc = tok.encode(test_str, add_bos=True, add_eos=True)
    dec = tok.decode(enc)
    print(f"    - Lossless Decode Test:  {dec == test_str} ({len(enc)} tokens)")

    # Archive to Drive
    if AURA_SESSION_CONFIG["MOUNT_DRIVE"] and DRIVE_PATHS["tokenizer"].exists():
        drive_tok_dir = DRIVE_PATHS["tokenizer"] / "target_32k"
        drive_tok_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(found_tok_dir / "tokenizer.json", drive_tok_dir / "tokenizer.json")
        shutil.copy2(found_tok_dir / "tokenizer_config.json", drive_tok_dir / "tokenizer_config.json")
else:
    print("[ERROR] Tokenizer target_32k not found. Please run Cell 7 to prepare it.")


## Cell 9: Stage C — Execute One Real Training Step (Validation Test)


In [ ]:
#@title Cell 9: Run Single-Step Verification Test { display-mode: "form" }
import time
import math
import torch
from aura.model.transformer import AuraModel
from aura.model.config import ModelConfig
from aura.data.dataset import MemoryMappedPretrainingDataset

current_mode = AURA_SESSION_CONFIG["MODE"]
print(f">>> [Cell 9 Active] Current Mode: {current_mode} | GPU Status: {'READY' if CAN_RUN_GPU else 'UNAVAILABLE'}")

SINGLE_STEP_PASSED = False

if current_mode in ["PREPARE", "PILOT", "RESUME"]:
    if not CAN_RUN_GPU:
        print("[SAFETY BLOCK] Skipping single-step test: CUDA GPU conditions not satisfied.")
    elif not DATASETS_READY:
        print("[SAFETY BLOCK] Skipping single-step test: 32k pretraining dataset not ready.")
    else:
        print("=" * 75)
        print("  SUSA AURA 1B — STAGE C: SINGLE-STEP GPU PIPELINE VALIDATION")
        print("=" * 75)

        device = torch.device("cuda:0")
        model_cfg = ModelConfig.from_yaml(PROFILE_CONFIG)

        print(f"Instantiating 1B model on {device} with gradient checkpointing...")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        model = AuraModel(model_cfg).to(device)
        model.gradient_checkpointing = True
        model.train()

        # Dataloader
        ds = MemoryMappedPretrainingDataset("data/train.bin", seq_len=model_cfg.max_seq_len)
        loader = torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False)
        batch = next(iter(loader))

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        # Configure Optimizer (8-bit AdamW)
        decay_params = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
        no_decay_params = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
        optim_groups = [
            {"params": decay_params, "weight_decay": 0.1},
            {"params": no_decay_params, "weight_decay": 0.0},
        ]

        if AURA_SESSION_CONFIG["USE_8BIT_ADAM"]:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(optim_groups, lr=3e-4, betas=(0.9, 0.95))
            print("Using bitsandbytes 8-bit AdamW optimizer.")
        else:
            optimizer = torch.optim.AdamW(optim_groups, lr=3e-4, betas=(0.9, 0.95))

        scaler = torch.amp.GradScaler("cuda", enabled=True)

        print("Executing forward pass under FP16 AMP...")
        start_t = time.time()
        with torch.amp.autocast("cuda", dtype=torch.float16):
            out = model(input_ids, labels=labels)
            loss = out.loss

        assert loss is not None and not torch.isnan(loss) and not torch.isinf(loss), "Loss was NaN or Inf!"
        print(f"Forward Pass Completed. Loss: {loss.item():.4f}")

        print("Executing backward pass and scaler step...")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()

        step_duration = time.time() - start_t
        actual_peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

        print(f"[✓] Single 1B Training Step Executed Successfully in {step_duration:.2f}s!")
        print(f"    - Initial Loss:      {loss.item():.4f}")
        print(f"    - Gradient Norm:     {grad_norm.item():.4f} (Gradients Non-Zero & Finite)")
        print(f"    - Actual Peak VRAM:  {actual_peak_vram:.2f} GiB (Safe within Tesla T4 14.56GB VRAM)")

        # Save verification checkpoint
        test_ckpt_path = Path("checkpoints/checkpoint_step_1.pt")
        test_ckpt_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({
            "step": 1,
            "total_tokens_processed": 2048,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": loss.item(),
            "model_config": model_cfg.to_dict(),
        }, test_ckpt_path)
        print(f"[✓] Saved verification checkpoint: {test_ckpt_path}")

        # Sync checkpoint to Google Drive
        if AURA_SESSION_CONFIG["MOUNT_DRIVE"] and DRIVE_PATHS["checkpoints"].exists():
            drive_dest = DRIVE_PATHS["checkpoints"] / "checkpoint_step_1.pt"
            shutil.copy2(test_ckpt_path, drive_dest)
            print(f"[✓] Checkpoint synced to Google Drive: {drive_dest}")

        SINGLE_STEP_PASSED = True
else:
    print(f"[INFO] Mode is '{current_mode}'. Single-step execution is skipped.")


## Cell 10: Stage D — Checkpoint Resume & State Integrity Verification


In [ ]:
#@title Cell 10: Verify Checkpoint Resume Integrity { display-mode: "form" }
current_mode = AURA_SESSION_CONFIG["MODE"]
print(f">>> [Cell 10 Active] Current Mode: {current_mode}")

ckpt_path = Path("checkpoints/checkpoint_step_1.pt")
if not ckpt_path.exists() and DRIVE_PATHS["checkpoints"].exists():
    drive_ckpt = DRIVE_PATHS["checkpoints"] / "checkpoint_step_1.pt"
    if drive_ckpt.exists():
        ckpt_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_ckpt, ckpt_path)

if current_mode in ["PREPARE", "PILOT", "RESUME"] and ckpt_path.exists():
    print("=" * 75)
    print("  SUSA AURA 1B — STAGE D: CHECKPOINT RESUME & RESTORATION TEST")
    print("=" * 75)

    print(f"Loading checkpoint into fresh CPU state: {ckpt_path}...")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    assert "model_state_dict" in checkpoint, "Missing model_state_dict in checkpoint!"
    assert "optimizer_state_dict" in checkpoint, "Missing optimizer_state_dict in checkpoint!"
    assert checkpoint.get("step") == 1, "Expected step 1 in saved checkpoint!"

    print("[✓] Checkpoint integrity verified:")
    print(f"    - Recorded Step:       {checkpoint['step']}")
    print(f"    - Recorded Loss:       {checkpoint['loss']:.4f}")
    print(f"    - Restored Tensors:    {len(checkpoint['model_state_dict'])} layers")
    print(f"    - Total Tokens:        {checkpoint.get('total_tokens_processed', 2048):,}")
    print("[✓] Resume test passed: Checkpoint format supports complete training state restoration.")
else:
    print(f"[INFO] Resume verification skipped (Mode: {current_mode}).")


## Cell 11: Stage E — 10-Step Pilot Run or Resumed Pretraining


In [ ]:
#@title Cell 11: Execute Pilot or Resumed Training { display-mode: "form" }
current_mode = AURA_SESSION_CONFIG["MODE"]
run_pilot_opt = AURA_SESSION_CONFIG["RUN_10_STEP_PILOT"]
print(f">>> [Cell 11 Active] Current Mode: {current_mode} | Pilot Opt-In: {run_pilot_opt}")

if current_mode == "PILOT":
    if not run_pilot_opt:
        print("[INFO] 10-Step Pilot skipped because RUN_10_STEP_PILOT=False.")
        print("       To run the 10-step pilot, check RUN_10_STEP_PILOT=True in Cell 1.")
    elif not CAN_RUN_GPU:
        print("[SAFETY BLOCK] Cannot run pilot: GPU conditions not met.")
    else:
        print("=" * 75)
        print("  SUSA AURA 1B — STAGE E: 10-STEP PILOT VALIDATION RUN")
        print("=" * 75)
        print("  Notice: This is a software pipeline validation across 10 steps.")
        print("          It is NOT meaningful pretraining.")
        print("=" * 75)

        pilot_cmd = [
            sys.executable, "scripts/train.py",
            "--config", AURA_SESSION_CONFIG["PROFILE_CONFIG"],
            "--train-bin", "data/train.bin",
            "--val-bin", "data/val.bin",
            "--pilot",
            "--device", "cuda:0"
        ]
        print(f"Executing: {' '.join(pilot_cmd)}")
        subprocess.run(pilot_cmd, check=True)
        print("[✓] 10-Step Pilot Execution Complete!")

elif current_mode == "RESUME":
    # Locate valid checkpoint
    target_ckpt = AURA_SESSION_CONFIG["RESUME_CHECKPOINT"]
    if not target_ckpt or target_ckpt == "<path_to_saved_checkpoint.pt>":
        # Auto-detect latest checkpoint
        available_ckpts = sorted(list(Path("checkpoints").glob("*.pt")) + list(DRIVE_PATHS["checkpoints"].glob("*.pt")))
        if available_ckpts:
            target_ckpt = str(available_ckpts[-1])
            print(f"[RESUME] Auto-selected newest available checkpoint: {target_ckpt}")
        else:
            raise FileNotFoundError(
                "[ERROR] RESUME FAILED: No saved checkpoint found on local disk or Drive. "
                "Please set MODE='PREPARE' or MODE='PILOT' in Cell 1 to train initial steps first."
            )

    print("=" * 75)
    print(f"  SUSA AURA 1B — RESUMING PRETRAINING FROM: {target_ckpt}")
    print("=" * 75)

    resume_cmd = [
        sys.executable, "scripts/train.py",
        "--config", AURA_SESSION_CONFIG["PROFILE_CONFIG"],
        "--train-bin", "data/train.bin",
        "--val-bin", "data/val.bin",
        "--resume-from", target_ckpt,
        "--additional-steps", str(AURA_SESSION_CONFIG["RESUME_ADDITIONAL_STEPS"]),
        "--device", "cuda:0"
    ]
    print(f"Executing: {' '.join(resume_cmd)}")
    subprocess.run(resume_cmd, check=True)
    print("[✓] Resumed Training Completed Successfully!")
else:
    print(f"[INFO] Stage E skipped (Mode: {current_mode}).")


## Cell 12: Stage F — Safe Stop, Google Drive Sync & Session Report


In [ ]:
#@title Cell 12: Sync Artifacts to Drive & Generate Run Report { display-mode: "form" }
import glob
import json
from datetime import datetime, timezone

current_mode = AURA_SESSION_CONFIG["MODE"]
print(f">>> [Cell 12 Active] Current Mode: {current_mode}")
print("=" * 75)
print("  SUSA AURA 1B — STAGE F: FINAL SESSION REPORT & ARTIFACT SYNC")
print("=" * 75)

# 1. Sync all checkpoints to persistent Google Drive
synced_count = 0
if AURA_SESSION_CONFIG["MOUNT_DRIVE"] and DRIVE_PATHS["checkpoints"].exists():
    local_ckpts = list(Path("checkpoints").glob("*.pt"))
    print(f"Scanning for local checkpoints to sync ({len(local_ckpts)} found)...")
    for ckpt in local_ckpts:
        dest = DRIVE_PATHS["checkpoints"] / ckpt.name
        if not dest.exists() or ckpt.stat().st_size != dest.stat().st_size:
            shutil.copy2(ckpt, dest)
            synced_count += 1
            print(f"  [SYNC] {ckpt.name} ({ckpt.stat().st_size / (1024**2):.2f} MB) -> {dest}")
    print(f"[✓] Google Drive checkpoint synchronization complete ({synced_count} synced).")

# 2. Write Run Report
report_data = {
    "project": "SUSA AURA",
    "architecture": "AURA 1B (target_1b)",
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "mode_executed": current_mode,
    "hardware": {
        "gpu_name": gpu_name,
        "gpu_vram_gb": round(gpu_vram, 2),
        "cuda_available": cuda_available,
        "system_ram_gb": round(sys_ram_total, 2),
    },
    "model_stats": stats,
    "preflight_passed": CAN_RUN_GPU,
    "saved_checkpoints": [str(c) for c in Path("checkpoints").glob("*.pt")],
}

report_path = Path("logs/colab_run_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report_data, f, indent=2)

if AURA_SESSION_CONFIG["MOUNT_DRIVE"] and DRIVE_PATHS["reports"].exists():
    shutil.copy2(report_path, DRIVE_PATHS["reports"] / report_path.name)
    print(f"[✓] Session report saved to Drive: {DRIVE_PATHS['reports'] / report_path.name}")

print("=" * 75)
print("  SESSION COMPLETE!")
print("=" * 75)
print("  To resume training in your next Colab session:")
print("  1. In Cell 1, set MODE = 'RESUME'.")
print("  2. Leave RESUME_CHECKPOINT blank to auto-detect the latest Drive checkpoint, or specify its path.")
print("  3. Run all cells. Training will continue from your exact step without restarting.")
print("=" * 75)
